# Module 2: Factor Modeling & Fama-French Risk Decomposition

- **Foundations of Factor Investing**: Shifting from single-factor CAPM to multi-factor Arbitrage Pricing Theory (APT) and empirical pricing anomalies.
- **Factor Data Pipeline**: Direct automated retrieval and parsing of Kenneth French's benchmark factor library (Market, Size, and Value).
- **Single-Asset Factor Regression**: Estimating market ($eta_{\text{Mkt}}$), size ($eta_{\text{SMB}}$), and value ($eta_{\text{HML}}$) loadings via OLS; examining how "apparent alpha" dissolves into factor exposures.
- **Cross-Sectional Style Tilts**: Multi-asset factor attribution across growth, value, large-cap, and small-cap equities on a 2D Style Exposure Map.
- **Dynamic Rolling Betas**: Tracking temporal factor drift, style migration, and macro regime shifts across 24-month rolling windows.
- **Factor Attribution & Factor-Neutral Hedging**: Decomposing returns into systematic factor contributions vs. idiosyncratic alpha, and constructing a multi-factor hedged portfolio.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import yfinance as yf
import urllib.request
import zipfile
import io
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

### 1. Conceptual Framework: CAPM vs. Fama-French Multi-Factor Model

- The **Capital Asset Pricing Model (CAPM)** (Sharpe, 1964; Lintner, 1965) asserts that an asset's expected return is solely a linear function of its exposure to broad market risk:
  $$R_{i, t} - R_{f, t} = \alpha_i + \beta_{i, \text{Mkt}} (R_{m, t} - R_{f, t}) + \epsilon_{i, t}$$
- **Empirical Pricing Anomalies**: Real-world equity returns systematically violate single-factor CAPM. Small-cap firms historically outperform large-cap firms (the *size anomaly*), and high book-to-market "value" firms systematically outperform low book-to-market "growth" firms (the *value anomaly*).
- **Fama-French 3-Factor Model (1993)**: Eugene Fama and Kenneth French introduced a multi-factor specification that accounts for these persistent dimensions of risk:
  $$R_{i, t} - R_{f, t} = \alpha_i + \beta_{i, \text{Mkt}} (R_{m, t} - R_{f, t}) + \beta_{i, \text{SMB}} \text{SMB}_t + \beta_{i, \text{HML}} \text{HML}_t + \epsilon_{i, t}$$
  - **$R_m - R_f$ (Market Risk Premium)**: Excess return of the value-weighted market portfolio over the 1-month US Treasury Bill rate.
  - **SMB (Small Minus Big)**: Spread between the returns of small-cap portfolios and large-cap portfolios (reflecting financial fragility and liquidity constraints).
  - **HML (High Minus Low)**: Spread between the returns of high book-to-market (value) portfolios and low book-to-market (growth) portfolios (reflecting financial leverage and distress risks).

In [ ]:
# Download Kenneth French's benchmark factor library directly from Dartmouth
ff_url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip'
req = urllib.request.Request(ff_url, headers={'User-Agent': 'Mozilla/5.0'})

with urllib.request.urlopen(req) as resp:
    zf = zipfile.ZipFile(io.BytesIO(resp.read()))
    raw_csv = zf.read(zf.namelist()[0]).decode('utf-8')

# Split out the monthly factor series (discard annual aggregation section)
monthly_raw = raw_csv.split(' Annual Factors:')[0]
lines = [l for l in monthly_raw.splitlines() if l.strip()]
header_idx = [i for i, l in enumerate(lines[:10]) if 'Mkt-RF' in l][0]

#IMPORTANT
ff_factors = pd.read_csv(io.StringIO('\n'.join(lines[header_idx:])), index_col=0)
ff_factors.index = pd.to_datetime(ff_factors.index.astype(str).str.strip(), format='%Y%m') + pd.offsets.MonthEnd(0)

# Convert percent figures into decimal returns
for col in ff_factors.columns:
    ff_factors[col] = pd.to_numeric(ff_factors[col], errors='coerce') / 100.0

ff_factors = ff_factors.dropna().loc['2000-01-01':]

# Calculate cumulative compounding for each factor: Product(1 + R)
cum_factors = (1 + ff_factors[['Mkt-RF', 'SMB', 'HML']]).cumprod()

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(cum_factors.index, cum_factors['Mkt-RF'], label='Market Factor (Mkt - RF)', color='#1f77b4', lw=2.2)
ax.plot(cum_factors.index, cum_factors['SMB'], label='Size Factor (SMB: Small Minus Big)', color='#2ca02c', lw=1.8)
ax.plot(cum_factors.index, cum_factors['HML'], label='Value Factor (HML: High Minus Low)', color='#d62728', lw=1.8)

ax.set_title('Kenneth French Research Factors: Cumulative Growth of $1.00 (2000 - Present)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Cumulative Growth ($)', fontsize=11)
ax.set_yscale('log')
ax.axhline(1.0, color='black', ls='--', lw=1.0, alpha=0.7)
ax.legend(loc='upper left', frameon=True, fontsize=11)

plt.tight_layout()
plt.show()

### 2. Single-Stock Analysis: CAPM vs. Fama-French 3-Factor Regression

- Evaluates an individual stock (**AAPL**) across both single-factor CAPM and the Fama-French 3-Factor model.
- **Ordinary Least Squares (OLS) Estimators**:
  - Excess Return Formulation: $R_{\text{AAPL}, t}^e = R_{\text{AAPL}, t} - R_{f, t}$
  - Single-Factor CAPM:
    $$R_{\text{AAPL}, t}^e = \alpha_{\text{CAPM}} + \beta_{\text{Mkt}} (R_{m, t} - R_{f, t}) + \epsilon_t$$
  - Multi-Factor Fama-French:
    $$R_{\text{AAPL}, t}^e = \alpha_{\text{FF3}} + \beta_{\text{Mkt}} (R_{m, t} - R_{f, t}) + \beta_{\text{SMB}} \text{SMB}_t + \beta_{\text{HML}} \text{HML}_t + \epsilon_t$$
- **The "Alpha Disappearance" Phenomenon**:
  A stock often appears to possess significant positive alpha under CAPM. However, once we control for its exposure to size and growth/value factor tilts, that apparent alpha frequently diminishes or disappears entirely.

In [ ]:
# Download monthly adjusted prices for Apple (AAPL)
aapl_raw = yf.download('AAPL', start='2015-01-01', end='2025-01-01', auto_adjust=True, progress=False)['Close']
aapl_monthly = aapl_raw.resample('ME').last().pct_change().dropna()

# Align time index between Apple and the Fama-French factors
common_dates = aapl_monthly.index.intersection(ff_factors.index)
merged_data = pd.DataFrame({
    'AAPL': aapl_monthly.loc[common_dates],
    'Mkt-RF': ff_factors.loc[common_dates, 'Mkt-RF'],
    'SMB': ff_factors.loc[common_dates, 'SMB'],
    'HML': ff_factors.loc[common_dates, 'HML'],
    'RF': ff_factors.loc[common_dates, 'RF']
})

# Calculate Apple's excess return over the risk-free rate
#IMPORTANT
merged_data['AAPL_Excess'] = merged_data['AAPL'] - merged_data['RF']

# 1. Fit Single-Factor CAPM Regression via OLS
X_capm = sm.add_constant(merged_data['Mkt-RF'])
#IMPORTANT
ols_capm = sm.OLS(merged_data['AAPL_Excess'], X_capm).fit()

# 2. Fit Multi-Factor Fama-French 3-Factor Regression via OLS
X_ff3 = sm.add_constant(merged_data[['Mkt-RF', 'SMB', 'HML']])
#IMPORTANT
ols_ff3 = sm.OLS(merged_data['AAPL_Excess'], X_ff3).fit()

# Compile side-by-side comparison table
model_comparison = pd.DataFrame({
    'Metric / Parameter': [
        'Monthly Alpha (α)', 'Annualized Alpha (%)', 'Alpha t-stat', 'Alpha p-value',
        'Market Beta (β_Mkt)', 'Size Beta (β_SMB)', 'Value Beta (β_HML)',
        'R-squared', 'Adjusted R-squared', 'Residual Std Error'
    ],
    'CAPM Model': [
        f"{ols_capm.params['const']:.4f}",
        f"{((1 + ols_capm.params['const'])**12 - 1) * 100:.2f}%",
        f"{ols_capm.tvalues['const']:.2f}",
        f"{ols_capm.pvalues['const']:.4f}",
        f"{ols_capm.params['Mkt-RF']:.3f}",
        "---",
        "---",
        f"{ols_capm.rsquared:.3f}",
        f"{ols_capm.rsquared_adj:.3f}",
        f"{np.sqrt(ols_capm.mse_resid):.4f}"
    ],
    'Fama-French 3-Factor Model': [
        f"{ols_ff3.params['const']:.4f}",
        f"{((1 + ols_ff3.params['const'])**12 - 1) * 100:.2f}%",
        f"{ols_ff3.tvalues['const']:.2f}",
        f"{ols_ff3.pvalues['const']:.4f}",
        f"{ols_ff3.params['Mkt-RF']:.3f}",
        f"{ols_ff3.params['SMB']:.3f}",
        f"{ols_ff3.params['HML']:.3f}",
        f"{ols_ff3.rsquared:.3f}",
        f"{ols_ff3.rsquared_adj:.3f}",
        f"{np.sqrt(ols_ff3.mse_resid):.4f}"
    ]
})

# Plot Actual vs. Fitted Excess Returns and Residual Distribution
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [1.2, 1]})

# Actual vs FF3 Fitted
ax1.plot(merged_data.index, merged_data['AAPL_Excess'] * 100, label='Actual AAPL Excess Return (%)', color='#1f77b4', lw=1.8)
ax1.plot(merged_data.index, ols_ff3.fittedvalues * 100, label='FF3 Fitted Return (%)', color='#ff7f0e', ls='--', lw=1.8)
ax1.set_title('Single-Stock Factor Fit: AAPL Excess Return vs. Fama-French 3-Factor Model', fontsize=13, fontweight='bold')
ax1.set_ylabel('Monthly Excess Return (%)', fontsize=11)
ax1.legend(loc='upper left', frameon=True)

# Diagnostic stats text box
stat_box = (
    f"FF3 Alpha: {ols_ff3.params['const']*12*100:.2f}% p.a. (p={ols_ff3.pvalues['const']:.3f})\n"
    f"β_Mkt: {ols_ff3.params['Mkt-RF']:.2f} | β_SMB: {ols_ff3.params['SMB']:.2f} | β_HML: {ols_ff3.params['HML']:.2f}\n"
    f"Adj. R²: {ols_ff3.rsquared_adj:.3f}"
)
ax1.text(0.02, 0.15, stat_box, transform=ax1.transAxes, fontsize=10, va='bottom',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.9))

# Residual Spread
ax2.plot(merged_data.index, ols_ff3.resid * 100, color='#2ca02c', lw=1.5, label='Idiosyncratic Residual Return (ε_t)')
ax2.axhline(0, color='black', ls='--', lw=1.0)
ax2.axhline(ols_ff3.resid.std() * 100, color='red', ls=':', label='±1σ Residual Threshold')
ax2.axhline(-ols_ff3.resid.std() * 100, color='red', ls=':')
ax2.set_title('Model Residuals: Idiosyncratic Asset Return Stripped of Market, Size, and Value Risk', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date', fontsize=11)
ax2.set_ylabel('Residual (%)', fontsize=11)
ax2.legend(loc='lower left', frameon=True)

plt.tight_layout()
plt.show()

model_comparison

### 3. Cross-Sectional Factor Exposure Matrix (Style Tilts across Equities)

- Broadens factor decomposition to a diversified universe spanning different styles and market capitalizations:
  - **Mega-Cap Growth / Tech**: AAPL, MSFT, NVDA
  - **Value / Conglomerate / Energy**: BRK-B, JPM, XOM
  - **Defensive / Quality**: JNJ, PG
  - **Small-Cap Benchmark**: IWM (iShares Russell 2000 ETF)
- **Style Exposure Map**:
  - Plots Size Beta ($\beta_{\text{SMB}}$) on the horizontal axis ($x < 0$: Mega-Cap tilt, $x > 0$: Small-Cap tilt).
  - Plots Value Beta ($\beta_{\text{HML}}$) on the vertical axis ($y < 0$: Growth tilt, $y > 0$: Value tilt).
  - Bubbles are sized proportionally to Market Beta ($\beta_{\text{Mkt}}$).

In [ ]:
universe_tickers = ['AAPL', 'MSFT', 'NVDA', 'BRK-B', 'JPM', 'XOM', 'JNJ', 'PG', 'IWM', 'SPY']

universe_raw = yf.download(universe_tickers, start='2018-01-01', end='2025-01-01', auto_adjust=True, progress=False)['Close']
universe_monthly = universe_raw.resample('ME').last().pct_change().dropna()

common_universe_dates = universe_monthly.index.intersection(ff_factors.index)
aligned_factors = ff_factors.loc[common_universe_dates]

factor_results = []
X_matrix = sm.add_constant(aligned_factors[['Mkt-RF', 'SMB', 'HML']])

for t in universe_tickers:
    ret_series = universe_monthly.loc[common_universe_dates, t]
    excess_ret = ret_series - aligned_factors['RF']
    
    #IMPORTANT
    res = sm.OLS(excess_ret, X_matrix).fit()
    
    alpha_annual = ((1 + res.params['const'])**12 - 1) * 100
    b_mkt = res.params['Mkt-RF']
    b_smb = res.params['SMB']
    b_hml = res.params['HML']
    adj_r2 = res.rsquared_adj
    p_alpha = res.pvalues['const']
    
    factor_results.append({
        'Ticker': t,
        'Annualized Alpha (%)': round(alpha_annual, 2),
        'Alpha p-val': f"{p_alpha:.3f}",
        'Market Beta (β_Mkt)': round(b_mkt, 2),
        'Size Beta (β_SMB)': round(b_smb, 2),
        'Value Beta (β_HML)': round(b_hml, 2),
        'Adj. R-squared': round(adj_r2, 3)
    })

factor_summary_df = pd.DataFrame(factor_results).sort_values('Adj. R-squared', ascending=False).reset_index(drop=True)

# 2D Factor Style Map
fig, ax = plt.subplots(figsize=(14, 8))
ax.axhline(0, color='black', ls='--', lw=1.2, alpha=0.7)
ax.axvline(0, color='black', ls='--', lw=1.2, alpha=0.7)

# Quadrant labels
ax.text(-0.55, 0.75, 'LARGE-CAP VALUE', fontsize=12, fontweight='bold', color='gray', alpha=0.6)
ax.text(0.35, 0.75, 'SMALL-CAP VALUE', fontsize=12, fontweight='bold', color='gray', alpha=0.6)
ax.text(-0.55, -0.75, 'LARGE-CAP GROWTH', fontsize=12, fontweight='bold', color='gray', alpha=0.6)
ax.text(0.35, -0.75, 'SMALL-CAP GROWTH', fontsize=12, fontweight='bold', color='gray', alpha=0.6)

# Scatter plot of assets in factor space
for _, row in factor_summary_df.iterrows():
    t = row['Ticker']
    x = row['Size Beta (β_SMB)']
    y = row['Value Beta (β_HML)']
    sz = max(row['Market Beta (β_Mkt)'] * 250, 80)
    
    color = '#1f77b4' if y < 0 else '#d62728'
    ax.scatter(x, y, s=sz, color=color, alpha=0.7, edgecolors='black', linewidth=1.5)
    ax.annotate(t, (x, y), xytext=(8, 5), textcoords='offset points', fontsize=11, fontweight='bold')

ax.set_title('Cross-Sectional Style Map: Factor Loadings across Equities (Size vs. Value)', fontsize=14, fontweight='bold')
ax.set_xlabel('Size Factor Loading: β_SMB (← Large Cap | Small Cap →)', fontsize=12)
ax.set_ylabel('Value Factor Loading: β_HML (← Growth | Value →)', fontsize=12)
ax.set_xlim(-0.7, 0.9)
ax.set_ylim(-0.9, 0.9)

plt.tight_layout()
plt.show()

factor_summary_df

### 4. Dynamic Rolling Factor Betas & Regime Shifts

- Factor exposures are not static over time. Companies evolve through business lifecycle shifts, product launches, capital allocation changes, and macroeconomic interest rate regimes.
- **24-Month Rolling OLS Regression**:
  $$R_{i, \tau}^e = \alpha_{\tau} + \beta_{\text{Mkt}, \tau} F_{\text{Mkt}, \tau} + \beta_{\text{SMB}, \tau} F_{\text{SMB}, \tau} + \beta_{\text{HML}, \tau} F_{\text{HML}, \tau} + \epsilon_{\tau}, \quad \tau \in [t-23, t]$$
- Tracking rolling factor trajectories exposes:
  - **Mega-Cap Drift**: How growing market capitalizations push $\beta_{\text{SMB}}$ progressively into deeply negative territory.
  - **Valuation Expansion / Compression**: Shifts between positive and negative $\beta_{\text{HML}}$ as market multiples expand during low-interest-rate environments and contract during rate-hiking cycles.

In [ ]:
window = 24  # 24 months rolling estimation window

dates_list = []
rolling_mkt = []
rolling_smb = []
rolling_hml = []
rolling_r2 = []

for i in range(window, len(merged_data)):
    window_df = merged_data.iloc[i-window:i]
    X_win = sm.add_constant(window_df[['Mkt-RF', 'SMB', 'HML']])
    y_win = window_df['AAPL_Excess']
    
    #IMPORTANT
    res_win = sm.OLS(y_win, X_win).fit()
    
    dates_list.append(window_df.index[-1])
    rolling_mkt.append(res_win.params['Mkt-RF'])
    rolling_smb.append(res_win.params['SMB'])
    rolling_hml.append(res_win.params['HML'])
    rolling_r2.append(res_win.rsquared_adj)

rolling_betas_df = pd.DataFrame({
    'Beta_Mkt': rolling_mkt,
    'Beta_SMB': rolling_smb,
    'Beta_HML': rolling_hml,
    'Adj_R2': rolling_r2
}, index=dates_list)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Rolling Market Beta
ax1.plot(rolling_betas_df.index, rolling_betas_df['Beta_Mkt'], color='#1f77b4', lw=2.0, label='24M Rolling Market Beta (β_Mkt)')
ax1.axhline(1.0, color='black', ls='--', lw=1.2, label='Market Neutral Beta (1.0)')
ax1.set_title('Dynamic Factor Evolution: Apple (AAPL) 24-Month Rolling Loadings', fontsize=13, fontweight='bold')
ax1.set_ylabel('Market Beta', fontsize=11)
ax1.legend(loc='upper right')

# Rolling Size Beta
ax2.plot(rolling_betas_df.index, rolling_betas_df['Beta_SMB'], color='#2ca02c', lw=2.0, label='24M Rolling Size Beta (β_SMB)')
ax2.axhline(0.0, color='black', ls='--', lw=1.2)
ax2.fill_between(rolling_betas_df.index, rolling_betas_df['Beta_SMB'], 0, color='#2ca02c', alpha=0.2)
ax2.set_ylabel('Size Beta (SMB)', fontsize=11)
ax2.legend(loc='upper right')

# Rolling Value Beta
ax3.plot(rolling_betas_df.index, rolling_betas_df['Beta_HML'], color='#d62728', lw=2.0, label='24M Rolling Value Beta (β_HML)')
ax3.axhline(0.0, color='black', ls='--', lw=1.2)
ax3.fill_between(rolling_betas_df.index, rolling_betas_df['Beta_HML'], 0, color='#d62728', alpha=0.2)
ax3.set_ylabel('Value Beta (HML)', fontsize=11)
ax3.set_xlabel('Date', fontsize=11)
ax3.legend(loc='upper right')

plt.tight_layout()
plt.show()

### 5. Factor Return Attribution & Factor-Neutral Hedging

- **Return Attribution Decomposition**:
  $$\bar{R}_i^e = \underbrace{\alpha_i}_{\text{True Alpha}} + \underbrace{\beta_{i, \text{Mkt}} \bar{F}_{\text{Mkt}}}_{\text{Market Contrib.}} + \underbrace{\beta_{i, \text{SMB}} \bar{F}_{\text{SMB}}}_{\text{Size Contrib.}} + \underbrace{\beta_{i, \text{HML}} \bar{F}_{\text{HML}}}_{\text{Value Contrib.}}$$
- **Factor-Neutral Portfolio Construction**:
  - Rather than being passively exposed to broad market swings, size bets, and value cycles, a quantitative manager hedges out the factor exposures:
    $$R_{\text{Factor-Neutral}, t} = R_{i, t}^e - \left(\beta_{i, \text{Mkt}} F_{\text{Mkt}, t} + \beta_{i, \text{SMB}} F_{\text{SMB}, t} + \beta_{i, \text{HML}} F_{\text{HML}, t}\right) = \alpha_i + \epsilon_{i, t}$$
  - Isolates pure idiosyncratic security selection, eliminating market timing and factor volatility.

In [ ]:
# Factor Return Attribution
mean_excess_aapl = merged_data['AAPL_Excess'].mean() * 12 * 100
contrib_alpha = ols_ff3.params['const'] * 12 * 100
contrib_mkt = ols_ff3.params['Mkt-RF'] * merged_data['Mkt-RF'].mean() * 12 * 100
contrib_smb = ols_ff3.params['SMB'] * merged_data['SMB'].mean() * 12 * 100
contrib_hml = ols_ff3.params['HML'] * merged_data['HML'].mean() * 12 * 100

attribution_df = pd.DataFrame({
    'Component': ['Pure Alpha (α)', 'Market Beta Contrib.', 'Size Beta Contrib.', 'Value Beta Contrib.', 'Total Realized Return'],
    'Annualized Return (%)': [contrib_alpha, contrib_mkt, contrib_smb, contrib_hml, mean_excess_aapl]
})

# Construct Factor-Neutral Return Series
#IMPORTANT
hedged_return = merged_data['AAPL_Excess'] - (
    ols_ff3.params['Mkt-RF'] * merged_data['Mkt-RF'] +
    ols_ff3.params['SMB'] * merged_data['SMB'] +
    ols_ff3.params['HML'] * merged_data['HML']
)

# Cumulative Performance Compounding
cum_unhedged = (1 + merged_data['AAPL_Excess']).cumprod()
cum_hedged = (1 + hedged_return).cumprod()
cum_market = (1 + merged_data['Mkt-RF']).cumprod()

# Peak-to-Trough Drawdowns
#IMPORTANT
dd_unhedged = (cum_unhedged / cum_unhedged.cummax() - 1) * 100
dd_hedged = (cum_hedged / cum_hedged.cummax() - 1) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(cum_unhedged.index, cum_unhedged, label='AAPL Unhedged Excess Return', color='#1f77b4', lw=2.2)
ax1.plot(cum_hedged.index, cum_hedged, label='AAPL Factor-Hedged (Pure Alpha + Residual)', color='#2ca02c', lw=2.2)
ax1.plot(cum_market.index, cum_market, label='Market Benchmark (Mkt - RF)', color='black', lw=1.5, ls='--')
ax1.set_title('Strategy Backtest: Raw Asset vs. Factor-Neutral Hedged Performance', fontsize=13, fontweight='bold')
ax1.set_ylabel('Portfolio Value ($)', fontsize=11)
ax1.legend(loc='upper left', frameon=True)

ax2.plot(dd_unhedged.index, dd_unhedged, label='AAPL Unhedged Drawdown (%)', color='#1f77b4', lw=1.5, ls='--')
ax2.fill_between(dd_hedged.index, dd_hedged, 0, color='#2ca02c', alpha=0.35, label='Factor-Hedged Drawdown (%)')
ax2.set_title('Drawdown Comparison: Volatility Dampening via Systematic Factor Hedging', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date', fontsize=11)
ax2.set_ylabel('Drawdown (%)', fontsize=11)
ax2.legend(loc='lower left', frameon=True)

plt.tight_layout()
plt.show()

# Performance Diagnostics Table
hedge_metrics_table = pd.DataFrame({
    'Portfolio / Strategy': ['AAPL Unhedged Excess', 'AAPL Factor-Hedged', 'Market Benchmark (Mkt-RF)'],
    'Annualized Return (%)': [
        round(merged_data['AAPL_Excess'].mean() * 12 * 100, 2),
        round(hedged_return.mean() * 12 * 100, 2),
        round(merged_data['Mkt-RF'].mean() * 12 * 100, 2)
    ],
    'Annualized Volatility (%)': [
        round(merged_data['AAPL_Excess'].std() * np.sqrt(12) * 100, 2),
        round(hedged_return.std() * np.sqrt(12) * 100, 2),
        round(merged_data['Mkt-RF'].std() * np.sqrt(12) * 100, 2)
    ],
    'Sharpe Ratio': [
        round((merged_data['AAPL_Excess'].mean() * 12) / (merged_data['AAPL_Excess'].std() * np.sqrt(12)), 2),
        round((hedged_return.mean() * 12) / (hedged_return.std() * np.sqrt(12)), 2),
        round((merged_data['Mkt-RF'].mean() * 12) / (merged_data['Mkt-RF'].std() * np.sqrt(12)), 2)
    ],
    'Max Drawdown (%)': [
        round(dd_unhedged.min(), 2),
        round(dd_hedged.min(), 2),
        round(((cum_market / cum_market.cummax() - 1) * 100).min(), 2)
    ]
})

hedge_metrics_table

### Module 2 Key Takeaways

- **Multi-Factor Risk Premia**: Single-factor CAPM fails to capture empirical equity return variations. The Fama-French 3-Factor model systematically incorporates size (SMB) and value (HML) risk premia.
- **Alpha Disaggregation**: Apparent "manager skill" or "stock outperformance" under CAPM is often revealed to be passive exposure to known systematic factor risk factors.
- **Dynamic Factor Loadings**: Factor betas drift over time as firms mature and macro monetary regimes shift, necessitating dynamic rolling calibration.
- **Factor-Neutral Engineering**: Isolating idiosyncratic alpha requires hedging out systematic market, size, and style factor bets to minimize unintended macroeconomic factor risk.